# Registro histórico: 09_focal_mixup

Origem: `C07` — `Colab Notebooks/Untitled23.ipynb`.

EfficientNet com Focal Loss, MixUp, OneCycleLR e busca de limiar.

**Limitações conhecidas:** Split estratificado por imagem; independência por lesão não demonstrada; logs de falha de multiprocessamento.

Cópia organizada em 25/10/2026. Código e texto originais preservados abaixo; saídas, contagens de execução e metadados do Colab removidos. As métricas salvas estão em `results/historical_evidence.json`. Não executar em lote: estes cadernos dependem do ambiente e de arquivos originais. Veja `docs/experiments.md` para o fluxo novo.


In [ ]:
import os
import random
import shutil
from pathlib import Path
from timeit import default_timer as timer

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import requests

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.models as models
from torchvision import transforms, datasets
from torchvision.models import EfficientNet_B0_Weights

In [ ]:
# ==============================================================================
# 1. Configuração e Download de Dados (Mantido)
# ==============================================================================

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Set random seeds for reproducibility
SEED = 91432
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Diretório onde o Kaggle vai salvar o dataset
DATASET_DIR = "data_ham10000"

os.makedirs(DATASET_DIR, exist_ok=True)

# Verifica se o arquivo de credenciais existe
kaggle_config_path = "kaggle.json"
print("kaggle.json existe?", os.path.exists(kaggle_config_path))

# Simulação de comandos para download do dataset (necessário para rodar o código)
# Estes comandos assumem que o arquivo kaggle.json está presente no ambiente.
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p {DATASET_DIR} --force

# Descompacta o arquivo
import zipfile
from glob import glob

zip_files = glob(os.path.join(DATASET_DIR, "*.zip"))
for zf in zip_files:
    print("Descompactando:", zf)
    with zipfile.ZipFile(zf, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)

import pandas as pd
import os

# Caminho do arquivo de metadados
metadata_path = os.path.join(DATASET_DIR, "HAM10000_metadata.csv")

# Lê o CSV de metadados
try:
    df = pd.read_csv(metadata_path)
except FileNotFoundError:
    print(f"ERRO: Arquivo de metadados não encontrado em {metadata_path}. Certifique-se de que o dataset foi baixado e descompactado corretamente.")
    # Cria um DataFrame de exemplo para evitar erro de código, mas o treinamento falhará
    df = pd.DataFrame({
        'image_id': ['ISIC_0000000', 'ISIC_0000001'],
        'dx': ['nv', 'mel']
    })


# Pastas onde estão as imagens
img_dir_1 = os.path.join(DATASET_DIR, "ham10000_images_part_1")
img_dir_2 = os.path.join(DATASET_DIR, "ham10000_images_part_2")


In [ ]:
# ==============================================================================
# 2. Pré-processamento e Divisão dos Dados
# ==============================================================================

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset

df_binary = df.copy()
df_binary["target"] = (df_binary["dx"] == "mel").astype(int)

def get_image_path(image_id):
    filename = image_id + ".jpg"
    path1 = os.path.join(img_dir_1, filename)
    path2 = os.path.join(img_dir_2, filename)
    if os.path.exists(path1):
        return path1
    elif os.path.exists(path2):
        return path2
    else:
        return None

df_binary["filepath"] = df_binary["image_id"].apply(get_image_path)
df_binary = df_binary.dropna(subset=["filepath"]).reset_index(drop=True)

train_df, temp_df = train_test_split(
    df_binary,
    test_size=0.3,
    random_state=SEED,
    stratify=df_binary["target"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["target"]
)

print(f"Train samples: {len(train_df)} (Melanoma: {train_df['target'].sum()})")
print(f"Val samples: {len(val_df)} (Melanoma: {val_df['target'].sum()})")
print(f"Test samples: {len(test_df)} (Melanoma: {test_df['target'].sum()})")

In [ ]:
# ==============================================================================
# 3. NOVA LOSS FUNCTION - FOCAL LOSS
# ==============================================================================

class FocalLoss(nn.Module):
    """Focal Loss para datasets desbalanceados"""
    def __init__(self, alpha=0.75, gamma=2.0, pos_weight=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, inputs, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(
            inputs, targets, reduction='none', pos_weight=self.pos_weight
        )
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

# Calcular pos_weight
y_train = train_df["target"].values
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
pos_weight_value = class_weights[1] * 3.0
pos_weight = torch.tensor(pos_weight_value, dtype=torch.float).to(device)
print(f"Peso da classe positiva (melanoma): {pos_weight_value:.4f}")

In [ ]:
# ==============================================================================
# 4. Dataset e Transformações APRIMORADAS
# ==============================================================================

class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["filepath"]
        label = row["target"]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float)

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4

weights = EfficientNet_B0_Weights.IMAGENET1K_V1
transform = weights.transforms()
mean = transform.mean
std = transform.std

# TRANSFORMAÇÕES APRIMORADAS
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),  # Aumentado
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    # transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    # transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

train_dataset = SkinLesionDataset(train_df, transform=train_transform)
val_dataset = SkinLesionDataset(val_df, transform=val_test_transform)
test_dataset = SkinLesionDataset(test_df, transform=val_test_transform)

In [ ]:
# ==============================================================================
# 5. WEIGHTED RANDOM SAMPLER para balanceamento
# ==============================================================================

class_counts = train_df['target'].value_counts()
class_weights_dict = {0: 1.0/class_counts[0], 1: 1.0/class_counts[1]}
sample_weights = train_df['target'].map(class_weights_dict).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,  # Usa sampler ao invés de shuffle
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [ ]:
# ==============================================================================
# 6. Modelo COM MAIS CAMADAS DESCONGELADAS
# ==============================================================================

base_model = models.efficientnet_b0(weights=weights)

# Descongelar MAIS camadas desde o início
for param in base_model.features[-12:].parameters():  # Aumentado para 12 camadas
    param.requires_grad = True

num_ftrs = base_model.classifier[1].in_features
base_model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features=num_ftrs, out_features=1)
)

base_model = base_model.to(device)

In [ ]:
# ==============================================================================
# 7. MIXUP IMPLEMENTATION
# ==============================================================================

def mixup_data(x, y, alpha=0.4):
    """Mixup augmentation"""
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
# ==============================================================================
# 8. Otimizador e Loss Function OTIMIZADOS
# ==============================================================================

# Learning rate MAIOR para convergência mais rápida
optimizer = optim.AdamW(base_model.parameters(),
                       lr=5e-4,  # AUMENTADO de 1e-4 para 5e-4
                       weight_decay=1e-4)

# Scheduler mais simples e eficaz
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=5e-4,  # Learning rate máximo
    epochs=50,
    steps_per_epoch=len(train_dataloader),
    pct_start=0.1,  # 10% do treinamento para warmup
    div_factor=10,  # Fator de divisão do LR
    final_div_factor=1000
)

# Loss function mais equilibrada
loss_fn = FocalLoss(alpha=0.8, gamma=3.0, pos_weight=pos_weight)  # Gamma reduzido

In [ ]:
# ==============================================================================
# 9. Funções de Treinamento COM MIXUP
# ==============================================================================

def train_step(model, data_loader, loss_fn, optimizer, device, use_mixup=False):
    model.train()
    train_loss, train_acc = 0, 0

    for X, y in tqdm(data_loader, desc="Training", leave=False):
        X, y = X.to(device), y.to(device)

        # Mixup apenas ocasionalmente e com alpha menor
        if use_mixup and random.random() > 0.8:  # Apenas 20% dos batches
            X, y_a, y_b, lam = mixup_data(X, y, alpha=0.2)  # Alpha reduzido
            y_pred_logits = model(X).squeeze(1)
            loss = mixup_criterion(loss_fn, y_pred_logits, y_a, y_b, lam)

            y_pred_probs = torch.sigmoid(y_pred_logits)
            y_pred_class = (y_pred_probs >= 0.5).float()
            acc = (y_pred_class == y_a).float().mean().item() * 100
        else:
            # Treinamento normal (mais direto)
            y_pred_logits = model(X).squeeze(1)
            loss = loss_fn(y_pred_logits, y)

            y_pred_probs = torch.sigmoid(y_pred_logits)
            y_pred_class = (y_pred_probs >= 0.5).float()
            acc = (y_pred_class == y).float().mean().item() * 100

        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping para treinamento mais estável
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()  # Step por batch no OneCycleLR

        train_loss += loss.item()
        train_acc += acc

    train_loss /= len(data_loader)
    train_acc /= len(data_loader)

    return train_loss, train_acc

def val_step(model, data_loader, loss_fn, device, threshold=0.5):
    model.eval()
    val_loss, val_acc = 0, 0
    all_probs, all_labels = [], []

    with torch.inference_mode():
        for X, y in tqdm(data_loader, desc="Validation", leave=False):
            X, y = X.to(device), y.to(device)

            y_pred_logits = model(X).squeeze(1)
            loss = loss_fn(y_pred_logits, y)

            y_pred_probs = torch.sigmoid(y_pred_logits)
            y_pred_class = (y_pred_probs >= threshold).float()

            acc = (y_pred_class == y).float().mean().item() * 100

            val_loss += loss.item()
            val_acc += acc

            all_probs.extend(y_pred_probs.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    val_loss /= len(data_loader)
    val_acc /= len(data_loader)

    # Calcular F1 para melanoma
    y_pred = (np.array(all_probs) >= threshold).astype(int)
    y_true = np.array(all_labels)

    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_melanoma = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return val_loss, val_acc, f1_melanoma, all_probs, all_labels


In [ ]:
# ==============================================================================
# 10. FUNÇÃO PARA ENCONTRAR THRESHOLD ÓTIMO
# ==============================================================================

from sklearn.metrics import f1_score

def find_optimal_threshold_f1(model, data_loader, device):
    """Encontra threshold que maximiza F1 para melanoma"""
    model.eval()
    all_probs, all_labels = [], []

    with torch.inference_mode():
        for X, y in data_loader:
            X, y = X.to(device), y.to(device)
            y_pred_logits = model(X).squeeze(1)
            y_pred_probs = torch.sigmoid(y_pred_logits)
            all_probs.extend(y_pred_probs.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    best_f1 = 0
    best_threshold = 0.5

    # Testar thresholds mais baixos para melanoma (melhor recall)
    for threshold in np.arange(0.1, 0.6, 0.02):
        y_pred = (np.array(all_probs) >= threshold).astype(int)
        f1 = f1_score(all_labels, y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    return best_threshold, best_f1

In [ ]:
# ==============================================================================
# 11. Loop de Treinamento OTIMIZADO
# ==============================================================================

EPOCHS = 150  # Reduzido pois OneCycleLR é mais eficiente
PATIENCE = 30
MIN_DELTA = 0.005  # Aumentado para ser menos sensível

best_f1 = 0.0
best_model_state = None
epochs_no_improve = 0

train_losses, train_accs = [], []
val_losses, val_accs, val_f1s = [], [], []

print("\n" + "="*60)
print("TREINAMENTO ACELERADO - FOCADO EM F1 > 0.75")
print("="*60)

train_start_time = timer()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 40)

    train_loss, train_acc = train_step(
        model=base_model,
        data_loader=train_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device,
        use_mixup=True
    )

    val_loss, val_acc, val_f1, val_probs, val_labels = val_step(
        model=base_model,
        data_loader=val_dataloader,
        loss_fn=loss_fn,
        device=device,
        threshold=0.3  # Threshold mais baixo inicialmente para melhor recall
    )

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    val_f1s.append(val_f1)

    current_lr = optimizer.param_groups[0]['lr']

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print(f"Val F1-Melanoma: {val_f1:.4f}")
    print(f"LR: {current_lr:.2e}")

    # Early stopping baseado em F1 com tolerância
    if val_f1 > best_f1 + MIN_DELTA:
        best_f1 = val_f1
        best_model_state = base_model.state_dict().copy()
        epochs_no_improve = 0
        print(f"🎯 NOVO MELHOR F1: {best_f1:.4f}")

        # Salvar checkpoint imediatamente quando melhora
        torch.save(best_model_state, f"best_model_f1_{best_f1:.4f}.pth")
    else:
        epochs_no_improve += 1
        print(f"⏳ Sem melhoria significativa: {epochs_no_improve}/{PATIENCE}")

    # Parada antecipada se atingir F1 desejado
    if val_f1 >= 0.75:
        print(f"🎉 F1-Score ALVO ATINGIDO: {val_f1:.4f}")
        break

    if epochs_no_improve >= PATIENCE:
        print(f"🛑 Early stopping ativado na época {epoch+1}")
        print(f"Melhor F1 alcançado: {best_f1:.4f}")
        break

    # Relatório de progresso a cada 5 épocas
    if (epoch + 1) % 5 == 0:
        print(f"\n📊 PROGRESSO - Época {epoch+1}:")
        print(f"   Melhor F1: {best_f1:.4f}")
        print(f"   F1 atual: {val_f1:.4f}")
        print(f"   Acurácia treino: {train_acc:.2f}%")
        print(f"   Acurácia val: {val_acc:.2f}%")

train_end_time = timer()

In [ ]:
# ==============================================================================
# 12. ENCONTRAR THRESHOLD ÓTIMO
# ==============================================================================

if best_model_state is not None:
    base_model.load_state_dict(best_model_state)
    print("\n📊 Buscando threshold ótimo no conjunto de validação...")
    optimal_threshold, optimal_f1_val = find_optimal_threshold(base_model, val_dataloader, device)

In [ ]:
# ==============================================================================
# 13. Avaliação Final COM THRESHOLD ÓTIMO
# ==============================================================================

def eval_model(model, data_loader, loss_fn, device, threshold=0.5):
    """Avaliação completa do modelo"""
    model.eval()
    test_loss, test_acc = 0, 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.inference_mode():
        progress_bar = tqdm(data_loader, desc="Evaluating")
        for X, y in progress_bar:
            X, y = X.to(device), y.to(device)

            y_pred_logits = model(X).squeeze(1)
            loss = loss_fn(y_pred_logits, y)

            y_pred_probs = torch.sigmoid(y_pred_logits)
            y_pred_class = (y_pred_probs >= threshold).float()

            correct = (y_pred_class == y).sum().item()
            acc = (correct / len(y)) * 100

            test_loss += loss.item()
            test_acc += acc

            all_preds.extend(y_pred_class.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(y_pred_probs.cpu().numpy())

            progress_bar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{acc:.2f}%'})

    test_loss /= len(data_loader)
    test_acc /= len(data_loader)

    return {
        "model_name": model.__class__.__name__,
        "model_loss": test_loss,
        "model_acc": test_acc,
        "predictions": all_preds,
        "labels": all_labels,
        "probabilities": all_probs
    }

print(f"\n🧪 Avaliando no conjunto de teste com threshold = {optimal_threshold:.2f}...")
test_results = eval_model(
    model=base_model,
    data_loader=test_dataloader,
    loss_fn=loss_fn,
    device=device,
    threshold=optimal_threshold
)

print(f"\n🎯 Resultados Finais no Teste:")
print(f"Test Loss: {test_results['model_loss']:.4f}")
print(f"Test Accuracy: {test_results['model_acc']:.2f}%")

# Métricas detalhadas
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns

class_names = ["nonMel", "mel"]

y_true = test_results['labels']
y_pred = test_results['predictions']

# Calcular F1 final
final_f1 = f1_score(y_true, y_pred)
print(f"\n🏆 F1-Score Final no Teste: {final_f1:.4f}")

print("\n📈 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix (F1={final_f1:.4f})', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Plot F1 ao longo do treinamento
plt.figure(figsize=(10, 6))
plt.plot(val_f1s, label='Validation F1-Score', marker='o')
plt.axhline(y=0.75, color='r', linestyle='--', label='Target F1=0.75')
plt.xlabel('Epoch')
plt.ylabel('F1-Score')
plt.title('F1-Score Evolution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# ==============================================================================
# 14. Salvamento
# ==============================================================================

LOCAL_MODEL_PATH = Path("/content/models")
LOCAL_MODEL_PATH.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "efficientNet-melanoma-optimized.pth"
MODEL_SAVE_PATH_LOCAL = LOCAL_MODEL_PATH / MODEL_NAME

counter = 1
original_model_name_local = MODEL_SAVE_PATH_LOCAL.stem
while MODEL_SAVE_PATH_LOCAL.exists():
    MODEL_SAVE_PATH_LOCAL = LOCAL_MODEL_PATH / f"{original_model_name_local}_{counter}{MODEL_SAVE_PATH_LOCAL.suffix}"
    counter += 1

print(f"\n💾 Salvando modelo em: {MODEL_SAVE_PATH_LOCAL}")
torch.save(obj=base_model.state_dict(), f=MODEL_SAVE_PATH_LOCAL)

# Salvar histórico
import json

history = {
    "train_losses": train_losses,
    "val_losses": val_losses,
    "train_accs": train_accs,
    "val_accs": val_accs,
    "val_f1s": val_f1s,
    "best_val_f1": best_f1,
    "optimal_threshold": optimal_threshold,
    "test_f1": final_f1,
    "test_accuracy": test_results['model_acc'],
    "training_time": total_time,
    "epochs": len(train_losses)
}

HISTORY_SAVE_PATH_LOCAL = LOCAL_MODEL_PATH / "training_history_optimized.json"

with open(HISTORY_SAVE_PATH_LOCAL, "w") as f:
    json.dump(history, f, indent=2)

print("✅ Modelo e histórico salvos com sucesso!")
print(f"\n{'='*60}")
print(f"RESUMO FINAL:")
print(f"{'='*60}")
print(f"Melhor F1 Validação: {best_f1:.4f}")
print(f"F1 Final no Teste: {final_f1:.4f}")
print(f"Threshold Ótimo: {optimal_threshold:.2f}")
print(f"{'='*60}")